In [20]:
import os
import wave
import contextlib
import webrtcvad
import numpy as np
import sys
print(sys.executable)
!{sys.executable} -m pip install moviepy
from moviepy import VideoFileClip
from pydub import AudioSegment


C:\Users\claus\anaconda3\python.exe


In [24]:
def extract_audio(mp4_path, wav_path, sample_rate=16000):
    """Extrahiert Audio aus MP4 als WAV mit mono, 16bit PCM, 16kHz"""
    video = VideoFileClip(mp4_path)
    audio = video.audio
    audio.write_audiofile(wav_path, fps=sample_rate, nbytes=2, buffersize=2000, codec='pcm_s16le')

def convert_audio(input_path, output_path):
    audio = AudioSegment.from_file(input_path)
    audio = audio.set_channels(1)  # Mono
    audio = audio.set_frame_rate(16000)  # 16kHz
    audio = audio.set_sample_width(2)  # 16-bit PCM
    audio.export(output_path, format="wav")
    
    return read_wave(output_path)
    
def read_wave(path):
    with contextlib.closing(wave.open(path, 'rb')) as wf:     # Öffnet eine WAV-Datei im Lese-Modus ('rb' = read binary)
        assert wf.getnchannels() == 1         # Sicherstellen, dass es sich um eine Mono-Aufnahme handelt (1 Kanal)
        assert wf.getsampwidth() == 2        # Sicherstellen, dass die Abtasttiefe 16 Bit beträgt (2 Bytes pro Sample)
        assert wf.getframerate() == 16000        # Sicherstellen, dass die Abtastrate 16000 Hz beträgt
        pcm_data = wf.readframes(wf.getnframes())        # Liest alle PCM-Daten (Pulse-Code Modulation) aus der Datei
        return pcm_data, wf.getframerate()        # Gibt die rohen PCM-Daten sowie die Abtastrate zurück

    
def frame_generator(frame_duration_ms, audio, sample_rate):
    n = int(sample_rate * (frame_duration_ms / 1000.0) * 2)    # Berechnet die Anzahl Bytes pro Frame (2 Bytes pro Sample für 16-bit PCM)
    offset = 0 # Startposition im Byte-Array
    timestamp = 0.0  # Startzeit in Sekunden
    duration = (float(n) / sample_rate) / 2.0 # Dauer eines Frames in Sekunden

    
    while offset + n < len(audio): # Generiert Frames, solange noch genug Daten vorhanden sind
        yield audio[offset:offset + n], timestamp # Gibt ein Frame und seinen Zeitstempel zurück
        timestamp += duration # Aktualisiert den Zeitstempel und Offset für das nächste Frame
        offset += n


In [25]:
def vad_collector(sample_rate, frame_duration_ms, padding_duration_ms, vad, frames):
    num_padding_frames = int(padding_duration_ms / frame_duration_ms)
    ring_buffer = []
    triggered = False

    voiced_segments = []
    current_start = 0.0

    for frame, timestamp in frames:
        is_speech = vad.is_speech(frame, sample_rate)

        if is_speech and not triggered:
            triggered = True
            current_start = timestamp

        elif not is_speech and triggered:
            triggered = False
            end = timestamp
            voiced_segments.append((round(current_start, 2), round(end, 2)))

    # Falls am Ende noch Sprache aktiv ist
    if triggered:
        voiced_segments.append((round(current_start, 2), round(timestamp, 2)))

    return voiced_segments

In [28]:
def main(mp4_path, wav_path, output_path):
    
    #MP4 in WAV wandeln
    extract_audio(mp4_path, wav_path)

    #audio, sample_rate = read_wave(wav_path)
    audio, sample_rate = convert_audio(wav_path,output_path)
    vad = webrtcvad.Vad(2)  # 0–3 (je höher, desto strenger)
    ''' WebRTC Voice Activity Detection  - Die Aggressivitätsstufe des VAD-Algorithmus 
    liegt zwischen 0 (am wenigsten aggressiv) und 3 (am aggressivsten)
    3 - Erkennt auch leise oder undeutliche Sprache (aber mehr False Positives)
    0 - Erkennt Sprache nur, wenn sie sehr eindeutig ist
    '''

    frames = list(frame_generator(30, audio, sample_rate))
    segments = vad_collector(sample_rate, 30, 300, vad, frames)

    #os.remove(wav_path) #optional zum aufräumen

    output_file = "sprachpausen.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("Typ | Start - Ende | Dauer\n")
        print("Typ | Start - Ende | Dauer")

        full_segments = []
        last_end = 0.0

        for start, end in segments:
            # Pause vor dem aktuellen Sprachsegment
            if start > last_end:
                pause_duration = round(start - last_end, 2)
                full_segments.append(("P", last_end, start, pause_duration))

            speech_duration = round(end - start, 2)
            full_segments.append(("S", start, end, speech_duration))
            last_end = end

        # Optional: Pause nach dem letzten Sprachsegment (falls du Audio-Ende kennst)

        # Ausgabe
        for typ, start, end, dauer in full_segments:
            line = f"{typ}/ {start:.2f} - {end:.2f} | Dauer: {dauer:.2f} Sek."
            print(line) #DEBUG
            f.write(line + "\n")


#Beispielanwendung
''' ggf. sind sehr kurze Pausen, sprechen kleiner weniger ms auszublenden
    Pausen benötigen eine ausreichende Länge um überhaupt etwas einsprechen zu können
'''

if __name__ == "__main__":
    # Ersetze den Pfad hier mit deiner .mp4-Datei
    main(mp4_path="Film_MDR.mp4", wav_path = "Film_Audio.wav", output_path="Film_Audi2.wav")


MoviePy - Writing audio in Film_Audio.wav


MoviePy - Done.
Typ | Start - Ende | Dauer
P/ 0.00 - 0.24 | Dauer: 0.24 Sek.
S/ 0.24 - 8.22 | Dauer: 7.98 Sek.
P/ 8.22 - 8.73 | Dauer: 0.51 Sek.
S/ 8.73 - 13.38 | Dauer: 4.65 Sek.
P/ 13.38 - 13.59 | Dauer: 0.21 Sek.
S/ 13.59 - 15.93 | Dauer: 2.34 Sek.
P/ 15.93 - 16.02 | Dauer: 0.09 Sek.
S/ 16.02 - 16.71 | Dauer: 0.69 Sek.
P/ 16.71 - 16.89 | Dauer: 0.18 Sek.
S/ 16.89 - 19.62 | Dauer: 2.73 Sek.
P/ 19.62 - 20.43 | Dauer: 0.81 Sek.
S/ 20.43 - 22.14 | Dauer: 1.71 Sek.
P/ 22.14 - 22.77 | Dauer: 0.63 Sek.
S/ 22.77 - 22.89 | Dauer: 0.12 Sek.
P/ 22.89 - 22.92 | Dauer: 0.03 Sek.
S/ 22.92 - 25.98 | Dauer: 3.06 Sek.
P/ 25.98 - 26.25 | Dauer: 0.27 Sek.
S/ 26.25 - 26.46 | Dauer: 0.21 Sek.
P/ 26.46 - 26.76 | Dauer: 0.30 Sek.
S/ 26.76 - 28.23 | Dauer: 1.47 Sek.
P/ 28.23 - 28.41 | Dauer: 0.18 Sek.
S/ 28.41 - 29.01 | Dauer: 0.60 Sek.
P/ 29.01 - 29.04 | Dauer: 0.03 Sek.
S/ 29.04 - 30.99 | Dauer: 1.95 Sek.
P/ 30.99 - 31.11 | Dauer: 0.12 Sek.
S/ 31.11 - 31.26 | Dauer: 0.15 Sek.
P/ 31.26 - 31.32 | Dauer: 0.